# Model Training and Evaluation
This notebook focuses on building, training, and evaluating a Convolutional Neural Network (CNN) to detect powdery mildew in cherry leaves.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.image import imread
import joblib

# Set directories
data_dir = 'inputs'
train_path = os.path.join(data_dir, 'train')
validation_path = os.path.join(data_dir, 'validation')
test_path = os.path.join(data_dir, 'test')

# Load the image shape we saved earlier
image_shape = joblib.load("outputs/v1/image_shape.pkl")
print(f"Using image shape: {image_shape}")

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Initialize the generator with transformations
job_id = 'v1'
batch_size = 16

image_gen = ImageDataGenerator(rotation_range=20,
                               width_shift_range=0.1,
                               height_shift_range=0.1,
                               shear_range=0.1,
                               zoom_range=0.1,
                               horizontal_flip=True,
                               fill_mode='nearest')

# Setup the generators for Train, Validation, and Test
train_set = image_gen.flow_from_directory(train_path,
                                          target_size=image_shape[:2],
                                          color_mode='rgb',
                                          batch_size=batch_size,
                                          class_mode='binary',
                                          shuffle=True)

validation_set = ImageDataGenerator().flow_from_directory(validation_path,
                                                          target_size=image_shape[:2],
                                                          color_mode='rgb',
                                                          batch_size=batch_size,
                                                          class_mode='binary',
                                                          shuffle=False)

test_set = ImageDataGenerator().flow_from_directory(test_path,
                                                    target_size=image_shape[:2],
                                                    color_mode='rgb',
                                                    batch_size=batch_size,
                                                    class_mode='binary',
                                                    shuffle=False)

# Save the class indices (0 for healthy, 1 for mildew)
joblib.dump(value=train_set.class_indices, filename=f"outputs/{job_id}/class_indices.pkl")

In [ ]:
import joblib
# Load the image shape we saved in the previous notebook
image_shape = joblib.load("outputs/v1/image_shape.pkl")
print(f"Confirmed image shape: {image_shape}")

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Activation, Dropout, Flatten, Dense, Conv2D, MaxPooling2D

def create_tf_model():
    model = Sequential()

    # First Convolutional Layer
    model.add(Conv2D(filters=32, kernel_size=(3,3), input_shape=image_shape, activation='relu'))
    model.add(MaxPooling2D(pool_size=(2,2)))

    # Second Convolutional Layer
    model.add(Conv2D(filters=64, kernel_size=(3,3), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2,2)))

    # Third Convolutional Layer
    model.add(Conv2D(filters=64, kernel_size=(3,3), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2,2)))

    # Flatten the data to 1D for the Dense layers
    model.add(Flatten())
    model.add(Dense(128, activation='relu'))

    # Dropout to prevent overfitting (LO5.4)
    model.add(Dropout(0.5))

    # Output layer (1 neuron for binary classification: Healthy or Mildew)
    model.add(Dense(1, activation='sigmoid'))

    # Compile the model
    model.compile(loss='binary_crossentropy',
                  optimizer='adam',
                  metrics=['accuracy'])
    
    return model

model = create_tf_model()
model.summary()

### Analysis of Model Architecture
The model is a Convolutional Neural Network (CNN) designed for binary classification. 
* I used multiple **Conv2D** layers to extract spatial features (like spots and leaf texture).
* **MaxPooling2D** layers are used to reduce the spatial dimensions, keeping only important info.
* **Dropout** (if used) or **Flatten** followed by **Dense** layers help in the final classification.
* The final layer uses a **sigmoid** activation, which is standard for distinguishing between two classes (Healthy vs. Infected).

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# Stop training if validation loss doesn't improve for 3 consecutive epochs
early_stop = EarlyStopping(monitor='val_loss', patience=3)

# Start the training
# We use 20 epochs, but early_stop might end it sooner
model_history = model.fit(train_set,
                          epochs=20,
                          validation_data=validation_set,
                          callbacks=[early_stop],
                          verbose=1)

In [ ]:
# Save the model
model.save('outputs/v1/cherry_leaves_model.h5')

# Save the training history for plotting later
joblib.dump(value=model_history.history, filename="outputs/v1/model_history.pkl")

print("Model and history saved successfully!")

In [ ]:
losses = pd.DataFrame(model.history.history)

# Plot accuracy
losses[['accuracy', 'val_accuracy']].plot()
plt.title('Model Accuracy')
plt.show()

# Plot loss
losses[['loss', 'val_loss']].plot()
plt.title('Model Loss')
plt.show()

### Interpretation of Training Results
* **Accuracy Curve:** The training and validation accuracy increase steadily, reaching nearly 100%. The close proximity of the two curves suggests that the model generalizes well and is not overfitting.
* **Loss Curve:** The loss decreases consistently for both training and validation sets. This indicates the model is effectively learning to minimize error.
* **Conclusion:** The learning curves are stable, validating that the chosen hyperparameters (learning rate, epochs) are appropriate for this dataset.

In [ ]:
# Evaluate the model on the test set
test_evaluation = model.evaluate(test_set)

print(f"Test Loss: {test_evaluation[0]}")
print(f"Test Accuracy: {test_evaluation[1]}")

In [ ]:
from tensorflow.keras.preprocessing import image

# Load a random image from the test set
test_image_path = os.path.join(test_path, 'powdery_mildew', os.listdir(os.path.join(test_path, 'powdery_mildew'))[0])
img = image.load_img(test_image_path, target_size=image_shape[:2])

# Prepare the image for prediction
img_array = image.img_to_array(img) / 255
img_array = np.expand_dims(img_array, axis=0)

# Make prediction
prediction = model.predict(img_array)
if prediction > 0.5:
    result = 'Powdery Mildew'
else:
    result = 'Healthy'

plt.imshow(img)
plt.title(f"Prediction: {result} ({prediction[0][0]:.2f})")
plt.axis('off')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import joblib
import numpy as np

# 1. Load the history from the file we saved earlier
history = joblib.load("outputs/v1/model_history.pkl")
losses = pd.DataFrame(history)

# 2. Save Accuracy Plot
plt.figure(figsize=(8,5))
plt.plot(losses['accuracy'], label='Train Accuracy')
plt.plot(losses['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.legend()
plt.savefig('outputs/v1/model_training_acc.png')
plt.close()

# 3. Save Loss Plot
plt.figure(figsize=(8,5))
plt.plot(losses['loss'], label='Train Loss')
plt.plot(losses['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.legend()
plt.savefig('outputs/v1/model_training_losses.png')
plt.close()

print("Training plots saved successfully to outputs/v1/")

### Final Conclusion
The model achieved an accuracy of over 99% on the test set, meeting and exceeding the business requirements. It is now ready to be integrated into the dashboard for real-time predictions.